# Model Açıklamaları İçin Hata Ayıklama Testleri — Makale Yeniden Üretimi

**Özgün çalışma:** Julius Adebayo, Michael Muelly, Ilaria Liccardi ve Been Kim, *Debugging Tests for Model Explanations*, NeurIPS (2020), [arXiv:2011.05429v1](https://arxiv.org/pdf/2011.05429v1).

**Kapsam:** Kaynaktaki üç bozulma kategorisinde incelenen dört hata örneğini *küçük, özgün sentetik veriler üzerinde* yeniden ele alan kontrollü bir **kavramsal yeniden uygulama**. Bu defter, özgün Birds-vs-Dogs ve ImageNet veri setlerini, tüm XAI yöntemlerini, tam deney ölçeğini veya 54 katılımcılı insan araştırmasını yeniden üretmez. Çıktılar özgün makalenin sayıları olarak sunulmaz.

**Öğrenci:** Tania El Hawari — 20259258043.

**AI katkısı:** Taslak kod, deney planı ve rapor oluşturma yapay zekâ yardımıyla hazırlandı. Kaydedilmiş çıktılar defterin yürütülmesiyle elde edilir; başarı veya kullanıcı çalışması sonucu uydurulmaz.

## Seçenek 1 — Zafiyet/Manipülasyon Analizi

Bu ödev için **Seçenek 1 — Zafiyet/Manipülasyon Analizi** seçilmiştir. *Debugging Tests for Model Explanations*, veri, model veya test girdisi bozulduğunda özellik atıflarının hatayı ortaya çıkarıp çıkaramadığını sınadığı için bu seçeneğe uygundur. Görsel olarak ikna edici bir açıklamanın modeldeki kusuru gizleyebilmesi, incelenen zafiyettir.

Dört sentetik deney bu soruyu farklı açılardan ele alır. **Sahte arka plan** deneyi sınıfla ilişkilendirilen ipucunun tahminlere ve atıflara etkisini, **yanlış etiket** deneyi bozulmuş eğitim etiketlerinin atıflardan ayırt edilip edilemediğini inceler. **Kademeli ağırlık yeniden başlatma**, atıfların öğrenilmiş parametrelere duyarlılığını, **dağılım dışı girdi vekili** ise görüntü dönüşümlerinin atıflara etkisini değerlendirir. Bunlar özgün makalenin kavramsal uyarlamalarıdır. Örneğin ters arka plan testinde doğruluk düşmedi. Her müdahale, modelin kandırıldığını veya açıklamanın çöktüğünü göstermez.

## 1. Araştırma sorusu ve hipotezler

Makale, veri, model ve test zamanı kontaminasyonlarının açıklamalarla ayırt edilip edilemeyeceğini inceler. Buradaki analojiler:

- **H1 · Sahte arka plan ilişkisi (veri):** Eğitim sırasında sınıfla ilişkilendirilen arka plan ipucuna ağ aşırı bağımlılık geliştirebilir. Sadece arka plan içeren görüntülerde başarı ve karşıt arka plana duyarlılık kontrol edilir.
- **H2 · Yanlış etiketler (veri):** %10 oranında çevrilmiş eğitim etiketleri bir modelin açıklamalarını değiştirebilir; fakat harita benzerliği yanlış etiketleri güvenilir biçimde tespit etmeyebilir.
- **H3 · Parametre yeniden başlatma (model):** Son katmandan giriş katmanına kademeli yeniden başlatma, açıklamaların modele duyarlılığını sınar. Sonuçlar yönteme ve katmana göre değişir.
- **H4 · Dağılım dışı girdi (test zamanı):** Görsel olarak değiştirilmiş girdiler için açıklamaların değişmesi belirsizdir. Benzer görünen haritalar tek başına bir girdinin dağılım içi olduğunun göstergesi değildir.

Bu hipotezler **önceden belirlenmiş değerlendirme sorularıdır**; sayısal geçme eşiği veya özgün makalenin sonuçlarını tutturma zorunluluğu yoktur.

## 2. Kullanılan kütüphaneler ve deney ortamı

| Paket | İşlev |
|---|---|
| NumPy | Tohumlu sentetik girdi ve sayısal kontroller |
| pandas | CSV manifestosu ve ölçüm tabloları |
| Pillow | 32×32 gri tonlu PNG üretimi |
| PyTorch | CNN, eğitim ve gradyan tabanlı açıklamalar |
| SciPy | Sıra korelasyonu ve SSIM için yerel istatistikler |
| Matplotlib | Deney görselleri |
| IPython / ipykernel | Notebook yürütmesi |
| tabulate / Mistune | Markdown tablo biçimleme ve Markdown’dan HTML üretimi |

Deneyler, internet bağlantısı veya hazır indirilmiş ağırlık gerektirmez. HTML’de matematik gösterimi MathJax’i çevrimiçi yükler. İşlemci üzerinde çalışır.

In [1]:
from pathlib import Path
from copy import deepcopy
from hashlib import sha256
import json, html, platform, re, sys
import numpy as np
import pandas as pd
from PIL import Image, ImageDraw
import matplotlib.pyplot as plt
from scipy.ndimage import gaussian_filter
from scipy.stats import rankdata
import torch
from torch import nn
import torch.nn.functional as F

ROOT = Path.cwd()
assert ROOT.name == 'makale-yeniden-uretimi', 'Defteri makale-yeniden-uretimi klasöründe çalıştırın.'
DATA = ROOT/'data'; FIG = ROOT/'sekiller'
FIG.mkdir(exist_ok=True)
SEED=20261122; SIDE=32
torch.manual_seed(SEED); np.random.seed(SEED)
torch.set_num_threads(2)
CLASSES={0:'kuş simgesi',1:'köpek simgesi'}
SPLIT_N={'train':256,'validation':64,'test':96}

def show_save(fig, name):
    fig.savefig(FIG/name,dpi=140,bbox_inches='tight')
    plt.close(fig)

def accuracy(model,x,y):
    model.eval()
    with torch.no_grad():
        l=model(x)
        return float((l.argmax(1)==y).float().mean()),float(F.cross_entropy(l,y))

print('Python',sys.version.split()[0], '| torch',torch.__version__,'| numpy',np.__version__)

Python 3.13.15 | torch 2.15.0+cpu | numpy 2.5.3


## 3. Matematiksel temel ve değerlendirme tasarımı

İki sınıflı model $f_\theta(x)$, girdi $x\in[0,1]^{32\times32}$ için iki logit döndürür. Seçilmiş sınıf logiti $s_c(x)$ için girdi gradyanı $\nabla_x s_c(x)$, Gradient×Input $x\odot\nabla_x s_c(x)$, SmoothGrad gürültüyle bozulan girdilerdeki gradyanların ortalamasıdır. Integrated Gradients, sıfır referansından gerçek girdiye giden çizgi boyunca gradyanları birleştirir:

$$\mathrm{IG}_c(x)=x\odot\int_0^1\nabla_z s_c(z)\big|_{z=tx}\,dt.$$

Guided Backpropagation, ReLU geri geçişinde, yalnızca ileri geçiş girdisi ve üstten gelen gradyan pozitifse gradyanı geçirir. Grad-CAM, son evrişimsel katmandaki sınıf gradyanlarını uzamsal olarak ortalar ve pozitif bileşeni büyütür. Guided Grad-CAM, Grad-CAM ve Guided Backpropagation haritalarını çarpar. Bu yöntemler **açıklama sadakatinin kanıtı değil, karşılaştırma araçlarıdır**.

**Ölçütler:** sıralamaya dayalı Spearman korelasyonu, işaret atılmış mutlak Spearman ve SSIM. Sabit haritalarda korelasyon tanımsızdır ve `NaN` olarak saklanır. Her deneyin ölçüm örneklemi açıkça raporlanır. Makaledeki orijinal SSIM koşullarıyla bire bir uyumluluk iddia edilmez.

## 4. Veri setinin üretimi

**Sınıflar:** Gerçek kuş ve köpek fotoğrafı yerine, merkezi siluetleri farklı olan basit **kuş benzeri ve köpek benzeri çizimler** kullanılır. Bu, makalenin gerçek görüntü verilerini yeniden oluşturmaz. **Sahte ipucu:** yatay açık bantlı arka plan veya dikey bitki çizgisi benzeri arka plan. Normal eğitimde arka plan sınıftan bağımsızdır; sahte-ilişkili eğitimde %95 olasılıkla sınıfa uyar. Kontrol veri setleri aynı bölmedeki örnek tohumlarından türetilir. Arka plan ipucu çakıştığında, aynı bölmedeki bazı kontrol ve temel PNG’ler birebir aynıdır.

Üç standart bölme (256/64/96) ayrıdır; test örnekleri model seçiminde kullanılmaz. Çalışma ayrıca sahte ilişkili eğitim/test, ters arka plan testi, sadece arka plan ve OOD türevlerini dosyaya kaydeder. Her dosyanın yol, sınıf, koşul, bölme, tohum ve SHA-256 değeri `veri.csv` içine yazılır. Böylece herhangi bir dış veri bağımlılığı olmadan yeniden üretilebilir.

In [2]:
def render_icon(label, cue, seed, empty=False):
    rng=np.random.default_rng(seed)
    im=Image.new('L',(64,64),color=52 if cue==0 else 42)
    d=ImageDraw.Draw(im)
    if cue==0:
        for y in range(3,64,8):
            d.line((0,y,63,y),fill=int(rng.integers(88,115)),width=2)
    else:
        for x in range(4,64,9):
            d.line((x,0,x,63),fill=int(rng.integers(93,125)),width=2)
            d.line((x-1,14,x+3,9),fill=145,width=1)
    if not empty:
        geom_rng=np.random.default_rng(seed+999)
        dx=int(geom_rng.integers(-4,5));dy=int(geom_rng.integers(-4,5))
        if label==0:  # bird-like icon with body, head, beak and wing
            d.ellipse((17+dx,28+dy,43+dx,48+dy),fill=216,outline=10,width=2)
            d.ellipse((32+dx,20+dy,47+dx,34+dy),fill=229,outline=10,width=2)
            d.polygon([(46+dx,25+dy),(57+dx,29+dy),(46+dx,33+dy)],fill=242)
            d.polygon([(18+dx,35+dy),(32+dx,29+dy),(35+dx,43+dy)],fill=160,outline=20)
            d.ellipse((41+dx,25+dy,43+dx,27+dy),fill=5)
            d.line((26+dx,46+dy,24+dx,54+dy),fill=240,width=2)
            d.line((37+dx,46+dy,36+dx,54+dy),fill=240,width=2)
        else: # dog-like icon with torso, head, ears, legs and tail
            d.rounded_rectangle((15+dx,31+dy,46+dx,46+dy),radius=3,fill=225,outline=10,width=2)
            d.ellipse((34+dx,20+dy,51+dx,37+dy),fill=225,outline=10,width=2)
            d.polygon([(37+dx,23+dy),(37+dx,15+dy),(43+dx,22+dy)],fill=175,outline=10)
            d.polygon([(46+dx,22+dy),(53+dx,16+dy),(51+dx,28+dy)],fill=175,outline=10)
            d.line((15+dx,33+dy,9+dx,25+dy),fill=245,width=4)
            d.rectangle((20+dx,45+dy,24+dx,54+dy),fill=235)
            d.rectangle((38+dx,45+dy,42+dx,54+dy),fill=235)
            d.ellipse((45+dx,27+dy,47+dx,29+dy),fill=5)
    img=np.asarray(im.resize((SIDE,SIDE),Image.Resampling.LANCZOS),dtype=np.float32)
    img=np.clip(img+rng.normal(0,4,img.shape),0,255).astype('uint8')
    return img

def transform_ood(arr):
    # Deterministic, visually strong domain change; label semantics may be impaired.
    img=255-np.asarray(arr,dtype=np.uint8)
    img=np.rot90(img,1).copy()
    band=(np.indices(img.shape)[0]//2)%2
    return np.clip(0.72*img+0.28*band*255,0,255).astype('uint8')

records=[]
def write_png(arr,split,kind,idx,label,cue,seed):
    folder=DATA/kind/split;folder.mkdir(parents=True,exist_ok=True)
    image_id=f'{kind}-{split}-{idx:04d}'
    path=folder/(image_id+'.png')
    Image.fromarray(arr,'L').save(path)
    rel=path.relative_to(ROOT).as_posix()
    records.append(dict(image_id=image_id,path=rel,split=split,kind=kind,label=label,
                        class_name=CLASSES[label],cue=cue,seed=seed,sha256=sha256(path.read_bytes()).hexdigest()))

for split,n in SPLIT_N.items():
    for idx in range(n):
        label=idx%2; seed=SEED+({'train':10000,'validation':20000,'test':30000}[split])+idx
        rng=np.random.default_rng(seed+301)
        independent_cue=int(rng.integers(0,2))
        clean=render_icon(label,independent_cue,seed)
        write_png(clean,split,'synthetic',idx,label,independent_cue,seed)
        if split=='train':
            spur_cue=label if rng.random()<.95 else 1-label
            write_png(render_icon(label,spur_cue,seed),split,'spurious',idx,label,spur_cue,seed)
        if split=='test':
            write_png(render_icon(label,label,seed),split,'spurious',idx,label,label,seed)
            write_png(render_icon(label,1-label,seed),split,'counterfactual',idx,label,1-label,seed)
            write_png(render_icon(label,label,seed,empty=True),split,'background_only',idx,label,label,seed)
            write_png(transform_ood(clean),split,'ood',idx,label,independent_cue,seed)

manifest=pd.DataFrame(records)
manifest.to_csv(ROOT/'veri.csv',index=False,encoding='utf-8-sig')
assert manifest.image_id.is_unique and len(manifest)==1056
assert all((ROOT/r.path).is_file() and sha256((ROOT/r.path).read_bytes()).hexdigest()==r.sha256 for r in manifest.itertuples())

def load(kind,split):
    rows=manifest[(manifest.kind==kind)&(manifest.split==split)].reset_index(drop=True)
    pixels=np.stack([np.asarray(Image.open(ROOT/p),dtype=np.float32)/255 for p in rows.path])
    return (torch.from_numpy(pixels[:,None,:,:]),torch.tensor(rows.label.to_numpy(),dtype=torch.long),rows)

train_x,train_y,train_rows=load('synthetic','train')
val_x,val_y,_=load('synthetic','validation')
test_x,test_y,test_rows=load('synthetic','test')
spur_train_x,spur_train_y,_=load('spurious','train')
spur_test_x,spur_test_y,spur_test_rows=load('spurious','test')
cf_x,cf_y,cf_rows=load('counterfactual','test')
bg_x,bg_y,bg_rows=load('background_only','test')
ood_x,ood_y,ood_rows_manifest=load('ood','test')
for paired in (spur_test_rows, cf_rows, bg_rows, ood_rows_manifest):
    assert paired[['seed','label']].equals(test_rows[['seed','label']]), "Test çiftleri aynı tohum ve etiketi taşımalı."
fig,axes=plt.subplots(2,5,figsize=(12,5))
for row,label in enumerate((0,1)):
    idx=label
    for ax,image,title in zip(axes[row],[train_x[idx,0],spur_train_x[idx,0],spur_test_x[idx,0],bg_x[idx,0],ood_x[idx,0]],
                              ['Bağımsız arka plan','Sahte eğitim ipucu','İpucu uyumlu test','Yalnız arka plan','OOD dönüşümü']):
        ax.imshow(image,cmap='gray',vmin=0,vmax=1)
        ax.axis('off')
        if row==0:ax.set_title(title,fontsize=9)
    axes[row,0].set_ylabel(CLASSES[label])
show_save(fig,'01-veri.png')
print('Manifesto: ',len(manifest),'PNG; bölmeler:',manifest.groupby(['kind','split']).size().to_dict())

Manifesto:  1056 PNG; bölmeler: {('background_only', 'test'): 96, ('counterfactual', 'test'): 96, ('ood', 'test'): 96, ('spurious', 'test'): 96, ('spurious', 'train'): 256, ('synthetic', 'test'): 96, ('synthetic', 'train'): 256, ('synthetic', 'validation'): 64}


## 5. CNN mimarisi ve model eğitimi

32×32×1 → Conv(12,3×3) → ReLU → 2×2 havuz → Conv(20,3×3) → ReLU → 2×2 havuz → 1280 → Dense(48) → ReLU → Dense(2).

Üç ağ aynı başlangıç tohumuyla eğitilir: **normal** (bağımsız arka planlı), **sahte ilişkili** (%95 sınıf-arka plan uyumlu), **yanlış etiketli** (normal görüntülerin %10 eğitim etiketi çevrilmiş). Geçerli test etiketleri değiştirilmez. Modeller, **doğrulama kaybı** en düşük dönemde saklanan ağırlıklarla değerlendirilir; veri seti küçüktür ve genelleme iddiası sınırlıdır.

In [3]:
class IconCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1=nn.Conv2d(1,12,3,padding=1)
        self.conv2=nn.Conv2d(12,20,3,padding=1)
        self.fc=nn.Linear(20*8*8,48)
        self.head=nn.Linear(48,2)
    def forward(self,x,guided=False,features=False):
        activate=GuidedReLU.apply if guided else F.relu
        x=F.max_pool2d(activate(self.conv1(x)),2)
        a=activate(self.conv2(x))
        x=F.max_pool2d(a,2).flatten(1)
        logits=self.head(activate(self.fc(x)))
        return (logits,a) if features else logits

class GuidedReLU(torch.autograd.Function):
    @staticmethod
    def forward(ctx,x):
        ctx.save_for_backward(x)
        return x.clamp_min(0)
    @staticmethod
    def backward(ctx,upstream):
        x,=ctx.saved_tensors
        return upstream.clamp_min(0)*(x>0)

def fit(x,labels,vx,vy,seed,epochs=20):
    torch.manual_seed(seed); model=IconCNN()
    gen=torch.Generator().manual_seed(seed+15)
    loader=torch.utils.data.DataLoader(torch.utils.data.TensorDataset(x,labels),
                         shuffle=True,batch_size=32,generator=gen)
    opt=torch.optim.Adam(model.parameters(),lr=.003)
    best_loss=np.inf;best_state=None;hist=[]
    for epoch in range(1,epochs+1):
        model.train()
        for xx,yy in loader:
            opt.zero_grad();loss=F.cross_entropy(model(xx),yy);loss.backward();opt.step()
        tr_acc,tr_loss=accuracy(model,x,labels)
        va_acc,va_loss=accuracy(model,vx,vy)
        hist.append(dict(epoch=epoch,train_acc=tr_acc,train_loss=tr_loss,val_acc=va_acc,val_loss=va_loss))
        if va_loss<best_loss:
            best_loss=va_loss;best_state=deepcopy(model.state_dict())
    model.load_state_dict(best_state);model.eval()
    return model,pd.DataFrame(hist)

model,hist=fit(train_x,train_y,val_x,val_y,SEED)
spur_model,spur_hist=fit(spur_train_x,spur_train_y,val_x,val_y,SEED)
rng=np.random.default_rng(SEED+444)
flip_indices=np.sort(rng.choice(len(train_y),size=int(len(train_y)*.10),replace=False))
wrong_labels=train_y.clone();wrong_labels[flip_indices]=1-wrong_labels[flip_indices]
wrong_model,wrong_hist=fit(train_x,wrong_labels,val_x,val_y,SEED)

results=[]
def evaluate_and_store(model,name,kind,x,y):
    a,l=accuracy(model,x,y)
    results.append(dict(model=name,dataset=kind,accuracy=a,cross_entropy=l))
    return a
for name,net in [('normal',model),('spurious',spur_model),('wrong_label',wrong_model)]:
    evaluate_and_store(net,name,'independent_test',test_x,test_y)
    evaluate_and_store(net,name,'correlated_test',spur_test_x,spur_test_y)
    evaluate_and_store(net,name,'reversed_background',cf_x,cf_y)
    evaluate_and_store(net,name,'background_only',bg_x,bg_y)
    evaluate_and_store(net,name,'ood',ood_x,ood_y)
score_table=pd.DataFrame(results)
score_table.to_csv(ROOT/'model_scores.csv',index=False)
fig,axs=plt.subplots(1,2,figsize=(11,3.5))
for name,h in [('Normal',hist),('Sahte ipucu',spur_hist),('Yanlış etiket',wrong_hist)]:
    axs[0].plot(h.epoch,h.train_acc,label=name)
    axs[1].plot(h.epoch,h.val_acc,label=name)
axs[0].set(title='Eğitim doğruluğu',xlabel='Dönem',ylabel='Doğruluk',ylim=(0,1.03))
axs[1].set(title='Doğrulama doğruluğu',xlabel='Dönem',ylim=(0,1.03))
axs[0].legend(fontsize=8)
show_save(fig,'02-egitim.png')

scores=score_table.pivot(index='model',columns='dataset',values='accuracy')
fig,ax=plt.subplots(figsize=(9,3.6))
img=ax.imshow(scores.to_numpy(),cmap='Blues',vmin=0,vmax=1)
for row in range(len(scores)):
 for col in range(len(scores.columns)):
  ax.text(col,row,f'{scores.iloc[row,col]:.2f}',ha='center',va='center',color='black')
ax.set(xticks=range(len(scores.columns)),xticklabels=list(scores.columns),
       yticks=range(len(scores)),yticklabels=list(scores.index),title='Test doğrulukları · açık kontrol koşulları')
plt.setp(ax.get_xticklabels(),rotation=25,ha='right',fontsize=8)
fig.colorbar(img,ax=ax,label='Doğruluk')
show_save(fig,'03-skorlar.png')
print(score_table.to_string(index=False,float_format=lambda x:f'{x:.3f}'))
print('Yanlış eğitim etiketi sayısı:',len(flip_indices))

      model             dataset  accuracy  cross_entropy
     normal    independent_test     1.000          0.000
     normal     correlated_test     1.000          0.000
     normal reversed_background     1.000          0.000
     normal     background_only     0.500          3.653
     normal                 ood     0.500          3.795
   spurious    independent_test     1.000          0.001
   spurious     correlated_test     1.000          0.000
   spurious reversed_background     1.000          0.002
   spurious     background_only     0.500          1.774
   spurious                 ood     0.500          4.568
wrong_label    independent_test     1.000          0.083
wrong_label     correlated_test     1.000          0.059
wrong_label reversed_background     1.000          0.105
wrong_label     background_only     0.500          0.813
wrong_label                 ood     0.500          1.545
Yanlış eğitim etiketi sayısı: 25


## 6. Açıklama haritaları ve benzerlik ölçütleri

Tüm karşılaştırmalarda **aynı girdinin aynı gerçek sınıf logiti** açıklanır; tahminin değişmesi, hedef sınıfın kendiliğinden değişmesine yol açmaz. 16 dengeli test örneği ve yanlış etiket için 16 bozulmuş eğitim örneği önceden seçilir. IG için 16 trapez aralığı, SmoothGrad için sabit tohumlu 8 gürültü örneği ($\sigma=0{,}08$) kullanılır. Sonuçlar yöntem ve deney bazında tanımlı ölçüm sayılarıyla özetlenir.

In [4]:
METHODS=['Gradient','Gradient × Input','Integrated Gradients','SmoothGrad','Guided Backprop','Grad-CAM','Guided Grad-CAM']
STEPS=16;NOISE_SAMPLES=8

def grad(model,inputs,target,guided=False):
    x=inputs.detach().requires_grad_(True)
    logits=model(x,guided=guided)[:,int(target)].sum()
    return torch.autograd.grad(logits,x)[0].detach()

def explain(net,image,target,salt):
    x=image.unsqueeze(0)
    g=grad(net,x,target)
    gb=grad(net,x,target,True)
    t=torch.linspace(0,1,STEPS+1).view(-1,1,1,1)
    path_g=grad(net,t*x,target)
    ig=x*((path_g[1:]+path_g[:-1])/2).mean(0,keepdim=True)
    rnd=torch.Generator().manual_seed(SEED+int(salt)+1500)
    noise=.08*torch.randn((NOISE_SAMPLES,*x.shape[1:]),generator=rnd)
    smooth=grad(net,x+noise,target).mean(0,keepdim=True)
    logits,features=net(x,features=True)
    d=torch.autograd.grad(logits[0,int(target)],features)[0]
    weights=d.mean((-2,-1),keepdim=True)
    cam=(features*weights).sum(1,keepdim=True).relu()
    cam=F.interpolate(cam.detach(),size=(SIDE,SIDE),mode='bilinear',align_corners=False)
    maps=[g,g*x,ig,smooth,gb,cam,cam*gb]
    out={method:z[0,0].detach().numpy() for method,z in zip(METHODS,maps)}
    assert all(a.shape==(SIDE,SIDE) and np.isfinite(a).all() for a in out.values())
    return out

def ssim(a,b):
    a,b=np.asarray(a,float),np.asarray(b,float)
    scale=max(np.abs(a).max(),np.abs(b).max(),1e-12)
    a=.5+.5*a/scale;b=.5+.5*b/scale
    ma,mb=gaussian_filter(a,1.5),gaussian_filter(b,1.5)
    va=gaussian_filter(a*a,1.5)-ma*ma
    vb=gaussian_filter(b*b,1.5)-mb*mb
    cov=gaussian_filter(a*b,1.5)-ma*mb
    return float(np.mean(((2*ma*mb+1e-4)*(2*cov+9e-4))/((ma*ma+mb*mb+1e-4)*(va+vb+9e-4))))

def rho(a,b):
    x,y=np.asarray(a,dtype=float).ravel(),np.asarray(b,dtype=float).ravel()
    if np.ptp(x)<1e-12 or np.ptp(y)<1e-12:return float('nan')
    return float(np.corrcoef(rankdata(x),rankdata(y))[0,1])

def compare(a,b):
    return dict(spearman=rho(a,b),spearman_abs=rho(abs(a),abs(b)),ssim=ssim(a,b))

probe=np.arange(SIDE*SIDE).reshape(SIDE,SIDE).astype(float)
assert np.isclose(ssim(probe,probe),1)
assert np.isclose(rho(probe,-probe),-1)
assert np.isnan(rho(np.ones_like(probe),probe))
panel=[int(i) for label in (0,1) for i in np.flatnonzero(test_y.numpy()==label)[:8]]
normal_maps={i:explain(model,test_x[i],test_y[i],i) for i in panel}

sample=panel[0]
fig,axes=plt.subplots(2,4,figsize=(11,5))
axes.flat[0].imshow(test_x[sample,0],cmap='gray',vmin=0,vmax=1);axes.flat[0].set_title('Girdi')
for ax,m in zip(axes.flat[1:],METHODS):
    a=normal_maps[sample][m];scale=max(abs(a).max(),1e-12)
    ax.imshow(a,cmap='RdBu_r',vmin=-scale,vmax=scale);ax.set_title(m,fontsize=8)
for ax in axes.flat:ax.axis('off')
show_save(fig,'04-yontemler.png')
print('Test paneli:',len(panel),'görüntü · yöntemler:',len(METHODS))

Test paneli: 16 görüntü · yöntemler: 7


## 7. Deney A — sahte korelasyonlu arka plan (veri kontaminasyonu)

Kaynak makalede kuş–köpek görüntülerinin sınıfa özgü arka planlarla eşlenmesi incelenir. Burada yalnızca **çizim benzeri örnekler** kullanılarak aynı hata türü kontrollü olarak taklit edilir. **Karşıt arka plan testi**, merkezi simgeyi sabit tutarken sınıf-arka plan ilişkisini tersine çevirir; **sadece arka plan testi** simgeyi kaldırır. Bu koşullarda model başarısı, atıf haritalarının arka plan üzerindeki mutlak kütle payıyla birlikte değerlendirilir. Dış çerçeve, nesne bölütleme maskesi değildir. Haritanın toplam mutlak atfı sıfırsa, bu veri çıktısında pay 0 olarak kaydedilir. Bu ölçüt tek başına nedensellik kanıtı değildir.

In [5]:
spur_maps={i:explain(spur_model,spur_test_x[i],test_y[i],i) for i in panel}
normal_same_maps={i:explain(model,spur_test_x[i],test_y[i],i) for i in panel}
# Background region is a conservative outer frame; foreground may occasionally extend outside it.
bg_mask=np.ones((SIDE,SIDE),bool);bg_mask[7:27,6:28]=False
background_rows=[]
for i in panel:
 for source,maps in [('normal',normal_same_maps),('spurious',spur_maps)]:
  for method in METHODS:
   a=abs(maps[i][method]);frac=float(a[bg_mask].sum()/(a.sum()+1e-12))
   background_rows.append(dict(image_id=str(spur_test_rows.loc[i,'image_id']),model=source,method=method,
                               abs_bg_fraction=frac))
background_df=pd.DataFrame(background_rows)
background_df.to_csv(ROOT/'background_attribution.csv',index=False)
fig,axes=plt.subplots(2,5,figsize=(12,5))
for row,i in enumerate(panel[:2]):
    axes[row,0].imshow(spur_test_x[i,0],cmap='gray',vmin=0,vmax=1)
    for col,method in enumerate(METHODS[:4],1):
        a=spur_maps[i][method];scale=max(abs(a).max(),1e-12)
        axes[row,col].imshow(a,cmap='RdBu_r',vmin=-scale,vmax=scale)
    for ax in axes[row]:ax.axis('off')
for ax,ttl in zip(axes[0],['Girdi',*METHODS[:4]]):ax.set_title(ttl,fontsize=8)
show_save(fig,'05-sahte-atif.png')

xaxis=np.arange(len(METHODS));wide=background_df.groupby(['model','method']).abs_bg_fraction.mean().unstack(0).reindex(METHODS)
fig,ax=plt.subplots(figsize=(11,3.5))
ax.bar(xaxis-.18,wide['normal'],width=.35,label='Normal model')
ax.bar(xaxis+.18,wide['spurious'],width=.35,label='Sahte ipucu modeli')
ax.set(xticks=xaxis,xticklabels=METHODS,ylabel='Dış çerçevedeki mutlak atıf oranı',ylim=(0,1),title='Arka plan önem payı (16 test örneği)')
plt.setp(ax.get_xticklabels(),rotation=22,ha='right',fontsize=8);ax.legend()
show_save(fig,'06-arka-plan-oran.png')
print('Ortalama dış çerçeve önem payı:',wide.round(3).to_dict())

Ortalama dış çerçeve önem payı: {'normal': {'Gradient': 0.464, 'Gradient × Input': 0.349, 'Integrated Gradients': 0.356, 'SmoothGrad': 0.464, 'Guided Backprop': 0.456, 'Grad-CAM': 0.231, 'Guided Grad-CAM': 0.224}, 'spurious': {'Gradient': 0.463, 'Gradient × Input': 0.346, 'Integrated Gradients': 0.344, 'SmoothGrad': 0.458, 'Guided Backprop': 0.5, 'Grad-CAM': 0.742, 'Guided Grad-CAM': 0.722}}


## 8. Deney B — yanlış etiketli eğitim örnekleri (veri kontaminasyonu)

Eğitim kümesindeki 256 örneğin önceden belirlenen 25'inin (%9,77) etiketleri karşı sınıfa çevrilir. Girdi görüntüleri değişmez. Aynı mimari, başlangıç tohumu, doğrulama verisi ve hedef sınıf kullanılır. Karşılaştırma **yanlış etiketlenmiş eğitim örneklerinde** yapılır. Haritaların benzer kalması, yanlış etiketleri tespit etmeyi garanti etmez. Hatalı etiketlerin test doğruluğuna etkisi de ayrıca gösterilir.

In [6]:
wrong_panel=list(map(int,flip_indices[:16]))
label_rows=[]; wrong_maps={}
for i in wrong_panel:
  original=explain(model,train_x[i],int(train_y[i]),SEED+i)
  altered=explain(wrong_model,train_x[i],int(train_y[i]),SEED+i)
  wrong_maps[i]=(original,altered)
  for method in METHODS:
   label_rows.append(dict(image_id=str(train_rows.loc[i,'image_id']),method=method,
                          **compare(original[method],altered[method])))
label_df=pd.DataFrame(label_rows); label_df.insert(0,'experiment','wrong_label')
fig,axes=plt.subplots(3,4,figsize=(10,6))
i=wrong_panel[0]
axes[0,0].imshow(train_x[i,0],cmap='gray',vmin=0,vmax=1);axes[0,0].set_title('Yanlış etiketli eğitim girdisi',fontsize=8)
axes[1,0].axis('off');axes[2,0].axis('off')
for col,method in enumerate(['Gradient','SmoothGrad','Guided Backprop'],1):
  a,b=wrong_maps[i][0][method],wrong_maps[i][1][method];scale=max(abs(a).max(),abs(b).max(),1e-12)
  axes[0,col].axis('off');axes[0,col].set_title(method,fontsize=8)
  for ax,v in [(axes[1,col],a),(axes[2,col],b)]:ax.imshow(v,cmap='RdBu_r',vmin=-scale,vmax=scale);ax.axis('off')
axes[1,0].text(0,.5,'Doğru eğitim\netiketi',va='center');axes[2,0].text(0,.5,'Bozulmuş eğitim\netiketi',va='center')
show_save(fig,'07-yanlis-etiket.png')
print(label_df.groupby('method')[['spearman','ssim']].mean().round(3).to_string())

                      spearman   ssim
method                               
Grad-CAM                 0.627  0.440
Gradient                 0.461  0.557
Gradient × Input         0.484  0.765
Guided Backprop          0.613  0.765
Guided Grad-CAM          0.604  0.736
Integrated Gradients     0.424  0.747
SmoothGrad               0.542  0.600


## 9. Deney C — parametre yeniden başlatma (model kontaminasyonu)

Eğitilmiş normal ağdan kopyalanan bir modelin ağırlıkları **head → fc → conv2 → conv1** sırasıyla, giderek biriken biçimde PyTorch'un başlangıç dağılımlarından yeniden örneklenir. Sabit test örnekleri ve sabit gerçek hedef sınıf kullanılır. Orijinal ağ ile değiştirilen ağların haritaları karşılaştırılır. Her yeniden başlatma aşaması bağımsız bir model eğitimi **değildir**; bu test *parametre duyarlılığı* ölçer. İyi bir görsel benzerlik yöntemin hatayı tespit ettiğini kanıtlamaz.

In [7]:
reinit_rows=[];last_maps=None
net=deepcopy(model)
for stage,module_name in enumerate(('head','fc','conv2','conv1'),1):
    torch.manual_seed(SEED+stage*19)
    getattr(net,module_name).reset_parameters()
    after={i:explain(net,test_x[i],int(test_y[i]),i) for i in panel}
    for i in panel:
        for method in METHODS:
            reinit_rows.append(dict(experiment='reinitialized_weights',stage=module_name,
                       n_layers_reset=stage,image_id=str(test_rows.loc[i,'image_id']),method=method,
                       **compare(normal_maps[i][method],after[i][method])))
    last_maps=after
reinit_df=pd.DataFrame(reinit_rows)
wide=reinit_df.groupby(['n_layers_reset','method']).spearman.mean().unstack('method').reindex(columns=METHODS)
fig,ax=plt.subplots(figsize=(9,4))
for method in METHODS:
    ax.plot(wide.index,wide[method],marker='o',label=method)
ax.set(xticks=[1,2,3,4],xticklabels=['head','+fc','+conv2','+conv1'],xlabel='Kademeli yeniden başlatma',ylabel='Ortalama Spearman',ylim=(-.5,1),title='Öğrenilmiş parametrelere duyarlılık')
ax.legend(ncol=2,fontsize=7)
show_save(fig,'08-parametre.png')

fig,axes=plt.subplots(2,4,figsize=(10,4.7))
idx=panel[0]
for col,m in enumerate(['Gradient','Gradient × Input','Guided Backprop','Grad-CAM']):
    ref,altered=normal_maps[idx][m],last_maps[idx][m]
    scale=max(abs(ref).max(),abs(altered).max(),1e-12)
    axes[0,col].set_title(m,fontsize=9)
    axes[0,col].imshow(ref,cmap='RdBu_r',vmin=-scale,vmax=scale)
    axes[1,col].imshow(altered,cmap='RdBu_r',vmin=-scale,vmax=scale)
for ax in axes.flat:ax.axis('off')
fig.text(.015,.73,'Eğitilmiş',rotation=90,fontsize=9,va='center')
fig.text(.015,.27,'Tümü sıfırdan',rotation=90,fontsize=9,va='center')
fig.subplots_adjust(left=.07)
show_save(fig,'09-parametre-harita.png')
print(wide.round(3).to_string())

method          Gradient  Gradient × Input  Integrated Gradients  SmoothGrad  Guided Backprop  Grad-CAM  Guided Grad-CAM
n_layers_reset                                                                                                          
1                  0.679             0.676                 0.648       0.702            0.916     0.970            0.918
2                 -0.037            -0.046                -0.049      -0.053            0.650     0.905            0.730
3                  0.019             0.022                 0.028       0.017            0.545     0.832            0.617
4                  0.014             0.012                 0.040       0.035            0.371     0.766            0.429


## 10. Deney D — dağılım dışı girdi (test zamanı kontaminasyonu)

Kaynak çalışmada Fashion-MNIST, MNIST, kuş–köpek ağı ve ImageNet ağları karşılaştırılır. Burada **gerçek alanlar arası veri kümesi bulunmadığı** için yalnızca kontrol amaçlı, görüntüyü ters renk, 90° dönüş ve bant dokusuyla değiştiren yapay bir OOD vekili kullanılır. Normal modelde özgün test girdisi ile dönüştürülmüş girdinin açıklamaları karşılaştırılır. Girdi pikselleri değiştiğinden bu benzerlik doğrudan açıklamanın OOD tanıma yeteneğini kanıtlamaz; sonuç yalnızca *görsel duyarlılık denemesi* olarak raporlanır. Haritalar ters döndürmeyle hizalanmaz ve kendi girdi koordinatlarında karşılaştırılır. Bu nedenle ölçümler, dönüşümün konumsal etkisini de içerir.

In [8]:
ood_rows=[];ood_maps={}
for i in panel:
    m=explain(model,ood_x[i],int(test_y[i]),i)
    ood_maps[i]=m
    for method in METHODS:
        ood_rows.append(dict(experiment='ood_proxy',image_id=str(test_rows.loc[i,'image_id']),
                             method=method,**compare(normal_maps[i][method],m[method])))
ood_df=pd.DataFrame(ood_rows)
fig,axes=plt.subplots(2,5,figsize=(12,5))
i=panel[0]
for row,(xx,m) in enumerate([(test_x[i],normal_maps[i]),(ood_x[i],ood_maps[i])]):
    axes[row,0].imshow(xx[0],cmap='gray',vmin=0,vmax=1)
    for col,method in enumerate(METHODS[:4],1):
        a=m[method];s=max(abs(a).max(),1e-12)
        axes[row,col].imshow(a,cmap='RdBu_r',vmin=-s,vmax=s)
    for ax in axes[row]:ax.axis('off')
for ax,name in zip(axes[0],['Girdi',*METHODS[:4]]):ax.set_title(name,fontsize=8)
show_save(fig,'10-ood.png')
print(ood_df.groupby('method')[['spearman','ssim']].mean().round(3).to_string())

                      spearman   ssim
method                               
Grad-CAM                 0.448  0.826
Gradient                 0.638  0.424
Gradient × Input         0.611  0.441
Guided Backprop          0.290  0.444
Guided Grad-CAM          0.204  0.692
Integrated Gradients     0.605  0.442
SmoothGrad               0.758  0.525


## 11. Bulguların sınırları ve orijinal makaleyle karşılaştırma

- **Özgün makalenin veri kontaminasyonu testi:** Gerçek kuş–köpek görüntülerinin arka planlarına sahte sinyaller yerleştirilir; yanlış eğitim etiketleri ayrıca incelenir. Bu defter gerçek görüntü değil çizim benzeri örnekler kullanır.
- **Özgün makalenin model kontaminasyonu testi:** Eğitimli VGG-16 ağının katmanları yeniden başlatılır. Burada çok daha küçük bir CNN kullanılır.
- **Özgün makalenin test zamanı kontaminasyonu testi:** Farklı veri alanları ve ağlar karşılaştırılır. Burada yalnızca kontrollü OOD görünüş değişikliği incelenir.
- **İnsan çalışması:** Makalede 54 kişilik çalışma vardır. **Bu projede insan çalışması yoktur.** Kullanıcı tepkileri, anket veya test sonucu uydurulmamıştır.

Sonuçlar küçük veri sayısı, tek eğitim tohumu, etiket bozulma örneklemi, basit sentetik çizimler, kullanılan 7 yöntem ve kontrol seçimine koşulludur. Bu proje kaynak makaleyi bire bir yeniden üretmez; seçilmiş hata türlerini izole eden doğrulanabilir bir eğitimsel yeniden uygulamadır.

**Gerçek veri / canlı kod** öğretim elemanının ayrı bir zorunluluğuysa bu defterin kaydedilmiş sentetik deneyi onun yerine geçmez.

In [9]:
# Run-derived tables and report; no manually entered experimental numbers.
all_similarity=pd.concat([
    label_df,
    reinit_df,
    ood_df
],ignore_index=True)
all_similarity.to_csv(ROOT/'benzerlik_sonuclari.csv',index=False)

def tr(x,places=3):
    if pd.isna(x):return 'tanımsız'
    return f'{float(x):.{places}f}'.replace('.',',')

def markdown_table(frame):
    return frame.to_markdown(index=False)

# Source of truth is generated measurements; HTML is regenerated from exactly this Markdown.
report=[]
def add(txt):report.append(txt.strip()+'\n\n')
add('# Model Açıklamaları İçin Hata Ayıklama Testleri — Makale Yeniden Üretimi')
add('**Tania El Hawari — 20259258043**  \n**Özgün yayın:** Julius Adebayo, Michael Muelly, Ilaria Liccardi ve Been Kim, *Debugging Tests for Model Explanations*, NeurIPS 2020, [arXiv:2011.05429v1](https://arxiv.org/pdf/2011.05429v1).')
add('> **Kapsam notu:** Bu çalışma, makaledeki üç bozulma kategorisindeki dört hata örneğine yönelik **küçük ve sentetik** bir kavramsal yeniden uygulamadır. Makalenin özgün veri setleri, mimarileri, istatistikleri veya 54 kişilik insan çalışması burada yeniden üretilmemiştir. Yapay zekâ destekli kodlama ve raporlama araçları kullanılmıştır.')
# Share the assignment rationale with the introductory Markdown cell.
notebook_cells = json.loads((ROOT / 'analiz.ipynb').read_text(encoding='utf-8'))['cells']
assignment_cell = next(cell for cell in notebook_cells if cell.get('id') == 'secenek-1')
add(''.join(assignment_cell['source']))
add('## 1. Araştırma amacı ve hipotezler')
add('H1: sınıfa bağlı arka plan bir sahte ilişki oluşturabilir. H2: yanlış etiketli eğitim örneklerinin haritaları benzer kalabilir. H3: rastgeleleştirilmiş ağırlıklar her açıklama yöntemini aynı şekilde etkilemez. H4: dağılım dışı görünüşe sahip girdi değişiminde harita benzerliği güvenilir OOD tanısı değildir.')
add('## 2. Kullanılan kütüphaneler')
add('Python, NumPy, pandas, Pillow, PyTorch, SciPy ve Matplotlib kullanıldı. Veriler ve ağırlıklar hazır XAI paketi kullanmadan oluşturuldu. Aynı kod `analiz.ipynb` içinde tekrar çalıştırılabilir.')
add('## 3. Matematiksel yöntem')
add(r'İki sınıflı bir CNN için $f_\theta(x)$ logitleri incelendi. Girdi gradyanı $\nabla_x s_c(x)$, Gradient × Input $x\odot\nabla_xs_c(x)$, Integrated Gradients, SmoothGrad, Guided Backprop, Grad-CAM ve Guided Grad-CAM hesaplandı. Sıralama korelasyonu Spearman ve görsel benzerlik SSIM ile ölçüldü. IG 16 trapez aralığı; SmoothGrad 8 gürültü örneği ve $\sigma=0{,}08$ kullandı. Aynı test girdisi için gerçek hedef sınıf logiti sabit kaldı.')
add('## 4. Veri üretimi ve kökeni')
add('Kuş ve köpek fotoğrafı yerine, merkezi şekilleri farklı **simge çizimleri** kullanıldı; arka plan yatay veya dikey dokulu olarak üretildi. Sınıf başına aynı sayıda örnek vardır. Bağımsız veri 256 eğitim, 64 doğrulama ve 96 test örneğinden oluşur. Sahte-ipuçlu eğitim kümesi ve test müdahaleleri ayrı dosyalarda saklanır. Toplam **'+str(len(manifest))+' PNG** için `veri.csv` içindeki SHA-256 karması doğrulandı. Tohum: '+str(SEED)+'.')
add('![Sentetik deney örnekleri](sekiller/01-veri.png)')
add('## 5. Model mimarisi ve eğitim')
add('32×32×1 → Conv(12) → ReLU/Havuz → Conv(20) → ReLU/Havuz → Dense(48) → Dense(2). Adam optimizasyonu, mini yığın boyutu 32, sabit tohum, eğitim üst sınırı 20 dönem. Normal, sahte ipuçlu ve %9,77 yanlış eğitim etiketli modellerin en iyi doğrulama kaybına sahip durumları kullanıldı.')
add('![Eğitim eğrileri](sekiller/02-egitim.png)')
add('## 6. Deney koşulları ve model doğrulukları')
ss=score_table.copy();ss['accuracy']=ss['accuracy'].map(tr);ss['cross_entropy']=ss['cross_entropy'].map(tr)
ss=ss.rename(columns={'model':'Model','dataset':'Test koşulu','accuracy':'Doğruluk','cross_entropy':'Çapraz entropi'})
add(markdown_table(ss))
add('![Test koşullarının doğruluk karşılaştırması](sekiller/03-skorlar.png)')
add('## 7. Atıf yöntemleri')
add('Gradient, Gradient × Input, Integrated Gradients, SmoothGrad, Guided Backprop, Grad-CAM ve Guided Grad-CAM. Pozitif ve negatif katkı işaretleri görselde korunur; sabit haritalar için korelasyon tanımsız olabilir. Yalnız görsel netlik, yöntemin modele sadakatini kanıtlamaz.')
add('![Yedi atıf yöntemi](sekiller/04-yontemler.png)')
add('## 8. Deney A · Sahte arka plan ilişkisi')
a=score_table[(score_table.model=='spurious')].set_index('dataset').accuracy
add('Sahte ilişkili eğitim modeli: uyumlu arka plan testi doğruluğu **'+tr(a['correlated_test'])+'**, ters arka plan testi **'+tr(a['reversed_background'])+'**, yalnız arka plan testi **'+tr(a['background_only'])+'**. Bu sayılar dışarıdan sağlanmış sonuçlar değil defterin hesapladığı ölçümlerdir. Dış çerçevedeki mutlak atıf payı, sadece betimleyici bir kontrol ölçütüdür.')
spur = score_table[score_table.model == 'spurious'].set_index('dataset')
delta_acc = a['correlated_test'] - a['reversed_background']
if delta_acc <= 0:
    interpretation = 'Ters arka plan doğruluğu düşmedi. Bu koşu sınıf kararlarında baskın arka plan bağımlılığı göstermiyor.'
else:
    interpretation = 'Ters arka plan doğruluğu düştü. Bu fark bu deney koşullarındaki arka plan duyarlılığını gösterir.'
add('**Kontrol yorumu:** Uyumlu ve ters arka plan doğruluklarının farkı **'+tr(delta_acc)+'**. '+interpretation+
    ' Yalnız arka plan testi de kendi doğruluk değeriyle birlikte değerlendirilmelidir. '
    'Ortalama çapraz entropi, uyumlu arka planda **'+tr(spur.loc['correlated_test','cross_entropy'],7)+
    '**, ters arka planda **'+tr(spur.loc['reversed_background','cross_entropy'],7)+
    '** oldu. Eşit doğruluk, sınıf skorlarının da aynı kaldığı anlamına gelmez. '
    'Model kontrol noktası bağımsız arka planlı doğrulama kümesindeki kayba göre seçildi. '
    'Dış çerçeve yaklaşık bir bölgedir ve özgün çalışmadaki GT-1/GT-2 maskeleriyle eşdeğer değildir.')
add('![Sahte arka plan atıfları](sekiller/05-sahte-atif.png)')
add('![Dış çerçeve atıf payı](sekiller/06-arka-plan-oran.png)')
add('## 9. Deney B · Yanlış eğitim etiketleri')
add(str(len(flip_indices))+' eğitim etiketinin yeri bozuldu; yanlış etiketli eğitim örneklerinden önceden belirlenen '+str(len(wrong_panel))+' tanesi açıklama karşılaştırmasında kullanıldı. Aşağıdaki betimleyici benzerlik ortalamaları doğru/bozulmuş etiket koşullarında aynı girdi ve gerçek sınıf için hesaplanmıştır. Bu sonuçlar hatalı etiketleri kesin olarak tespit etmez.')
wrong_train_acc,_=accuracy(wrong_model,train_x,wrong_labels)
wrong_true_acc,_=accuracy(wrong_model,train_x,train_y)
add('Yanlış etiketli ağın **bozulmuş eğitim etiketlerine** doğruluğu '+tr(wrong_train_acc)+'; aynı görüntülerin **gerçek etiketlerine** doğruluğu '+tr(wrong_true_acc)+'. Dolayısıyla bu modelin yanlış etiketleri mutlaka ezberlediği varsayılmamıştır. Kaynak makalenin yanlış etiket bulgusuyla doğrudan sayısal karşılaştırma yapılamaz.')
lab=label_df.groupby('method',as_index=False)[['spearman','ssim']].mean()
for col in ['spearman','ssim']:lab[col]=lab[col].map(tr)
add(markdown_table(lab.rename(columns={'method':'Yöntem','spearman':'Spearman','ssim':'SSIM'})))
add('![Yanlış etiket açıklamaları](sekiller/07-yanlis-etiket.png)')
add('## 10. Deney C · Kademeli parametre yeniden başlatma')
add('Modelde sırasıyla `head`, `fc`, `conv2`, `conv1` katmanları kademeli sıfırlandı. '+str(len(panel))+' test örneği üzerinde her aşama karşılaştırıldı. Gözlemler ağın bu özel sürümüne aittir, özgün VGG-16 sonucunu tekrarlamaz.')
r=reinit_df.groupby(['stage','method'],as_index=False)[['spearman','ssim']].mean()
for col in ['spearman','ssim']:r[col]=r[col].map(tr)
add(markdown_table(r.rename(columns={'stage':'Son sıfırlanan katman','method':'Yöntem','spearman':'Spearman','ssim':'SSIM'})))
add('![Katman duyarlılığı](sekiller/08-parametre.png)')
add('![Model yeniden başlatma haritaları](sekiller/09-parametre-harita.png)')
add('## 11. Deney D · OOD vekil testi')
add('Test görüntülerinin renkleri tersine çevrildi, 90° döndürüldü ve bantlı doku eklendi. Bu, orijinal makaledeki Fashion-MNIST / MNIST / ImageNet çapraz-model deneyinin aynısı değildir. Aynı modelle özgün ve bozulmuş görüntüler kıyaslandığından girdi değişikliği karşılaştırmayı etkiler. Haritalar ters döndürmeyle hizalanmadığı için ölçümler konumsal değişimi de içerir.')
o=ood_df.groupby('method',as_index=False)[['spearman','ssim']].mean()
for col in ['spearman','ssim']:o[col]=o[col].map(tr)
add(markdown_table(o.rename(columns={'method':'Yöntem','spearman':'Spearman','ssim':'SSIM'})))
add('![OOD vekil girdiler ve haritalar](sekiller/10-ood.png)')
add('## 12. Özgün makale ile karşılaştırma ve sınırlar')
add('Kaynak çalışma; sahte arka plan ve yanlış etiketli veri, VGG-16 parametre yeniden başlatma ve çeşitli alan dışı veri setleri üzerinde atıfları test eder; ayrıca 54 katılımcılı insan deneyi raporlar. Bu uygulama özgün fotoğrafları, tam atıf yöntemi kümesini, GT-1/GT-2 maske değerlendirmesini, çapraz veri kümesi ve model karşılaştırmalarını veya 54 katılımcılı insan deneyini yeniden üretmez. Tek eğitim tohumu ve tek yanlış etiket örneklemi kullanılmıştır. Şekillerin istatistiksel belirsizliği için güven aralığı verilmemiştir. Bu defterin sonuçları yalnızca bu deney koşullarında geçerlidir.')
add('## 13. Tekrar üretilebilirlik, AI katkısı ve sınırlama')
add('Veri yolu, tohum, sınıf, koşul ve SHA-256 bilgileri `veri.csv` içindedir. Sayısal sonuçlar `model_scores.csv`, `background_attribution.csv`, `benzerlik_sonuclari.csv` dosyalarında tutulur. Defterin son hücresi bu metin ve HTML dosyasını oluşturur. Yapay zekâ destekli araçlar kod, anlatım ve teknik kontrollerin hazırlanmasında kullanıldı. Kullanıcı kendi çalıştırmasını ve yorumlarını akademik gereklilikler çerçevesinde değerlendirmelidir. Makaleden alınmış sonuçların bu projede yeniden ölçüldüğü iddia edilmez.')

add('## 14. Tüm benzerlik ölçümleri (betimleyici özet)')
add('Aşağıdaki her satır bir deney, aşama, yöntem ve ölçüt içindir. NaN değerleri ortalama/standart sapma hesabına alınmadı; **tanımlı** ve **toplam** çift sayısı birlikte raporlanır. Çoklu yöntemler ve aynı görüntü tekrarları bağımsız örneklem gibi yorumlanmamalıdır.')
long_rows=[]
for group_key,chunk in all_similarity.groupby(['experiment','method'],dropna=False):
    exp,method=group_key
    if exp=='reinitialized_weights':
        stages=sorted(chunk.stage.dropna().unique().tolist())
    else:stages=['—']
    for stage in stages:
        sub=chunk if stage=='—' else chunk[chunk.stage==stage]
        for metric in ['spearman','spearman_abs','ssim']:
            v=sub[metric].dropna()
            long_rows.append({'Deney':exp,'Aşama':stage,'Yöntem':method,'Ölçüt':metric,
                              'Ortalama':tr(v.mean()) if len(v)>0 else 'tanımsız',
                              'Örnek ss':tr(v.std(ddof=1)) if len(v)>1 else 'tanımsız',
                              'Tanımlı çift':int(len(v)),'Toplam çift':int(len(sub))})
add(markdown_table(pd.DataFrame(long_rows)))


out=''.join(report)
(ROOT/'makale.md').write_text(out,encoding='utf-8')
try:
    import mistune
    renderer=mistune.create_markdown(plugins=['table'])
    body=renderer(out)
except ImportError as ex:
    raise RuntimeError('HTML için mistune gereklidir: pip install mistune') from ex
head="""<!doctype html><html lang="tr"><head><meta charset="utf-8"><meta name="viewport" content="width=device-width, initial-scale=1"><title>Model Açıklamaları İçin Hata Ayıklama Testleri</title><script>window.MathJax={tex:{inlineMath:[["$","$"]],displayMath:[["$$","$$"]]}};</script><script defer src="https://cdn.jsdelivr.net/npm/mathjax@3/es5/tex-chtml.js"></script><style>body{background:#f2f5f9;color:#18283d;font:16px/1.65 system-ui,sans-serif;margin:0}main{max-width:1040px;margin:0 auto;background:white;padding:2rem 3rem}h1,h2{color:#173657}h2{margin-top:2rem}img{display:block;max-width:100%;height:auto;margin:1rem auto}table{display:block;width:100%;overflow-x:auto;border-collapse:collapse;font-size:.9rem}th,td{border-bottom:1px solid #e1e6ed;padding:.45rem .6rem;text-align:left}th{background:#eaf0f7}blockquote{border-left:4px solid #537aa6;padding-left:1rem;color:#4f5e70}code{background:#edf1f5;padding:.1rem .2rem}a{color:#126094}@media(max-width:650px){main{padding:1rem}}</style></head><body><main>"""
(ROOT/'makale.html').write_text(head+body+'</main></body></html>',encoding='utf-8')
expected={f'{i:02d}' for i in range(1,11)}
assert expected=={p.stem[:2] for p in FIG.glob('*.png')}
assert all('sekiller/'+p.name in out for p in FIG.glob('*.png'))
assert len(manifest)==1056
print('Rapor hazır:',len(out),'karakter; 10 figür; 1056 PNG; ölçüm kayıtları:',len(all_similarity))

Rapor hazır: 28413 karakter; 10 figür; 1056 PNG; ölçüm kayıtları: 672
